# Flow Matching vs DDPM for Diffusion Policy: Results Analysis

**CS 8803 Deep Reinforcement Learning - Final Project**

This notebook reproduces key experimental results comparing Flow Matching (FM) and DDPM for Diffusion Policy on the PushT benchmark task.

## Overview

- **Baseline**: DDPM with 100 inference steps
- **Modification**: Flow Matching with 4-16 Euler ODE steps
- **Key Finding**: FM achieves 13-27× speedup with &lt;15% accuracy drop

## Contents
1. Load Experimental Results
2. Results Comparison Table
3. Latency vs Accuracy Analysis
4. Training Dynamics
5. Ablation Study: Inference Steps
6. Summary Statistics

In [ ]:
# Import Required Libraries
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style for publication-quality figures
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Define project root
PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
LOGS_DIR = PROJECT_ROOT / "logs" / "experiments"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Results Directory: {RESULTS_DIR}")
print(f"Logs Directory: {LOGS_DIR}")

## 1. Load Experimental Results

We load results from completed training experiments. Results are stored in JSON files after evaluation.

In [ ]:
# Experimental Results from Training Runs (Dec 2025)
# These values are extracted from training logs and evaluation

experiments = {
    "DDPM Baseline (100 steps)": {
        "method": "DDPM",
        "architecture": "UNet",
        "inference_steps": 100,
        "best_score": 0.869,
        "best_epoch": 450,
        "latency_p50_ms": 650.0,
        "jerk": None,
        "training_time_hrs": 20.0,
        "status": "completed"
    },
    "FM UNet (4 steps)": {
        "method": "FM",
        "architecture": "UNet", 
        "inference_steps": 4,
        "best_score": 0.7504,
        "best_epoch": 1050,
        "latency_p50_ms": 24.48,
        "jerk": 5563.7,
        "training_time_hrs": 6.5,
        "status": "completed"
    },
    "FM Steps=4": {
        "method": "FM",
        "architecture": "UNet",
        "inference_steps": 4,
        "best_score": 0.7566,
        "best_epoch": 1100,
        "latency_p50_ms": 24.39,
        "jerk": 5576.5,
        "training_time_hrs": 6.8,
        "status": "completed"
    },
    "FM Steps=8": {
        "method": "FM",
        "architecture": "UNet",
        "inference_steps": 8,
        "best_score": 0.7790,
        "best_epoch": 1300,
        "latency_p50_ms": 47.95,
        "jerk": 4696.1,
        "training_time_hrs": 8.0,
        "status": "completed"
    },
    "FM Steps=16": {
        "method": "FM",
        "architecture": "UNet",
        "inference_steps": 16,
        "best_score": 0.7568,
        "best_epoch": 1250,
        "latency_p50_ms": 150.70,
        "jerk": 4526.8,
        "training_time_hrs": 8.8,
        "status": "completed"
    },
    "FM Transformer (4 steps)": {
        "method": "FM",
        "architecture": "Transformer",
        "inference_steps": 4,
        "best_score": 0.0722,
        "best_epoch": 1700,
        "latency_p50_ms": 21.14,
        "jerk": 43501.4,
        "training_time_hrs": 7.2,
        "status": "completed"
    }
}

# Convert to DataFrame
df = pd.DataFrame(experiments).T
df.index.name = "Experiment"
df = df.reset_index()

print("Experiments Loaded:")
df

## 2. Main Results Comparison

### Table 1: DDPM vs Flow Matching Performance

In [ ]:
# Filter to main comparison experiments (UNet architecture only)
main_results = df[df['architecture'] == 'UNet'].copy()

# Calculate speedup relative to DDPM baseline
ddpm_latency = main_results[main_results['method'] == 'DDPM']['latency_p50_ms'].values[0]
main_results['speedup'] = ddpm_latency / main_results['latency_p50_ms']

# Calculate accuracy drop relative to DDPM
ddpm_score = main_results[main_results['method'] == 'DDPM']['best_score'].values[0]
main_results['accuracy_drop_%'] = (ddpm_score - main_results['best_score']) / ddpm_score * 100

# Display results table
display_cols = ['Experiment', 'method', 'inference_steps', 'best_score', 'latency_p50_ms', 'speedup', 'accuracy_drop_%']
results_table = main_results[display_cols].copy()
results_table.columns = ['Experiment', 'Method', 'Steps', 'Test Score', 'Latency (ms)', 'Speedup', 'Accuracy Drop (%)']

print("=" * 80)
print("Table 1: Flow Matching vs DDPM Performance Comparison")
print("=" * 80)
results_table.round(2)

## 3. Visualization: Latency vs Accuracy Trade-off

This figure demonstrates the key finding: Flow Matching achieves significant speedup with minimal accuracy loss.

In [ ]:
# Create figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Latency vs Score scatter
ax1 = axes[0]
colors = {'DDPM': '#e74c3c', 'FM': '#3498db'}
for method in main_results['method'].unique():
    data = main_results[main_results['method'] == method]
    ax1.scatter(data['latency_p50_ms'], data['best_score'], 
                s=150, c=colors[method], label=method, alpha=0.8, edgecolors='black')
    for _, row in data.iterrows():
        ax1.annotate(f"{int(row['inference_steps'])} steps", 
                    (row['latency_p50_ms'], row['best_score']),
                    textcoords="offset points", xytext=(5, 5), fontsize=9)

ax1.set_xlabel('Inference Latency (ms)', fontsize=12)
ax1.set_ylabel('Test Score', fontsize=12)
ax1.set_title('Figure 1: Latency vs Accuracy Trade-off', fontsize=14)
ax1.legend(title='Method', fontsize=10)
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)

# Plot 2: Bar chart comparison
ax2 = axes[1]
x = np.arange(len(main_results))
width = 0.35

bars1 = ax2.bar(x - width/2, main_results['best_score'], width, label='Test Score', color='#3498db')
ax2_twin = ax2.twinx()
bars2 = ax2_twin.bar(x + width/2, main_results['speedup'], width, label='Speedup', color='#e74c3c', alpha=0.7)

ax2.set_ylabel('Test Score', fontsize=12, color='#3498db')
ax2_twin.set_ylabel('Speedup (×)', fontsize=12, color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels([f"Steps={int(s)}" for s in main_results['inference_steps']], rotation=45, ha='right')
ax2.set_title('Figure 2: Score vs Speedup by Configuration', fontsize=14)
ax2.set_ylim(0, 1.0)
ax2_twin.set_ylim(0, 30)

# Add legend
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('../results/latency_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to results/latency_vs_accuracy.png")

## 4. Ablation Study: Effect of Inference Steps

This analysis shows how the number of inference steps affects both accuracy and latency for Flow Matching.

In [ ]:
# Filter FM experiments for inference steps ablation
fm_data = main_results[main_results['method'] == 'FM'].copy()
fm_data = fm_data.sort_values('inference_steps')

# Create ablation figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Score vs Steps
ax1 = axes[0]
ax1.plot(fm_data['inference_steps'], fm_data['best_score'], 'o-', color='#3498db', linewidth=2, markersize=10)
ax1.axhline(y=ddpm_score, color='#e74c3c', linestyle='--', linewidth=2, label=f'DDPM Baseline ({ddpm_score:.3f})')
ax1.set_xlabel('Inference Steps', fontsize=12)
ax1.set_ylabel('Test Score', fontsize=12)
ax1.set_title('(a) Score vs Inference Steps', fontsize=13)
ax1.legend()
ax1.set_xticks([4, 8, 16])
ax1.grid(True, alpha=0.3)

# Plot 2: Latency vs Steps
ax2 = axes[1]
ax2.plot(fm_data['inference_steps'], fm_data['latency_p50_ms'], 'o-', color='#2ecc71', linewidth=2, markersize=10)
ax2.axhline(y=ddpm_latency, color='#e74c3c', linestyle='--', linewidth=2, label=f'DDPM Baseline ({ddpm_latency:.0f}ms)')
ax2.set_xlabel('Inference Steps', fontsize=12)
ax2.set_ylabel('Latency (ms)', fontsize=12)
ax2.set_title('(b) Latency vs Inference Steps', fontsize=13)
ax2.legend()
ax2.set_xticks([4, 8, 16])
ax2.grid(True, alpha=0.3)

# Plot 3: Jerk (smoothness) vs Steps
ax3 = axes[2]
ax3.plot(fm_data['inference_steps'], fm_data['jerk'], 'o-', color='#9b59b6', linewidth=2, markersize=10)
ax3.set_xlabel('Inference Steps', fontsize=12)
ax3.set_ylabel('Mean Jerk (lower = smoother)', fontsize=12)
ax3.set_title('(c) Action Smoothness vs Inference Steps', fontsize=13)
ax3.set_xticks([4, 8, 16])
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/inference_steps_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to results/inference_steps_ablation.png")

# Print ablation table
print("\n" + "=" * 60)
print("Table 2: Inference Steps Ablation Study")
print("=" * 60)
ablation_table = fm_data[['Experiment', 'inference_steps', 'best_score', 'latency_p50_ms', 'jerk', 'speedup']].copy()
ablation_table.columns = ['Experiment', 'Steps', 'Score', 'Latency (ms)', 'Jerk', 'Speedup']
print(ablation_table.to_string(index=False))

## 5. Architecture Comparison: UNet vs Transformer

Flow Matching with Transformer architecture shows significantly worse performance, indicating architecture sensitivity.

In [ ]:
# Architecture comparison
arch_comparison = df[df['inference_steps'] == 4].copy()

print("=" * 60)
print("Table 3: Architecture Comparison (4 Inference Steps)")
print("=" * 60)

arch_table = arch_comparison[['Experiment', 'architecture', 'best_score', 'latency_p50_ms', 'jerk']].copy()
arch_table.columns = ['Experiment', 'Architecture', 'Score', 'Latency (ms)', 'Jerk']
print(arch_table.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#3498db', '#e74c3c']
x = np.arange(len(arch_comparison))
bars = ax.bar(x, arch_comparison['best_score'], color=colors)
ax.set_ylabel('Test Score', fontsize=12)
ax.set_title('Figure 3: Architecture Comparison (FM with 4 Steps)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(arch_comparison['architecture'])
ax.set_ylim(0, 1.0)

for bar, score in zip(bars, arch_comparison['best_score']):
    ax.annotate(f'{score:.3f}', 
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../results/architecture_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️ Key Finding: FM with Transformer performs poorly (0.072) compared to UNet (0.750)")
print("   This suggests Flow Matching may require architecture-specific tuning.")

## 6. Summary Statistics and Key Findings

In [ ]:
# Summary statistics
print("=" * 70)
print("SUMMARY: Flow Matching vs DDPM for Diffusion Policy")
print("=" * 70)

# Best FM result (8 steps)
best_fm = main_results[main_results['best_score'] == main_results[main_results['method'] == 'FM']['best_score'].max()].iloc[0]
ddpm_row = main_results[main_results['method'] == 'DDPM'].iloc[0]

print(f"\n📊 DDPM Baseline Performance:")
print(f"   - Test Score: {ddpm_row['best_score']:.3f}")
print(f"   - Latency: {ddpm_row['latency_p50_ms']:.0f} ms")
print(f"   - Inference Steps: {int(ddpm_row['inference_steps'])}")

print(f"\n🚀 Best Flow Matching Configuration (8 Steps):")
print(f"   - Test Score: {best_fm['best_score']:.3f}")
print(f"   - Latency: {best_fm['latency_p50_ms']:.1f} ms")
print(f"   - Speedup: {best_fm['speedup']:.1f}×")
print(f"   - Accuracy Drop: {best_fm['accuracy_drop_%']:.1f}%")

print(f"\n✅ KEY FINDINGS:")
print(f"   1. Flow Matching achieves 13.5× speedup with only 10.4% accuracy drop (8 steps)")
print(f"   2. 4 steps: 27× faster but 13% accuracy drop")
print(f"   3. 16 steps: Diminishing returns (slower than 8 steps with similar accuracy)")
print(f"   4. Optimal configuration: 8 inference steps")
print(f"   5. Architecture matters: Transformer FM fails (0.072 score)")

print("\n" + "=" * 70)
print("CONCLUSION: Flow Matching is a viable alternative to DDPM for real-time control")
print("=" * 70)

## 7. Export Results to JSON

Save all results to a consolidated JSON file for the submission.

In [ ]:
# Export consolidated results
consolidated_results = {
    "project": "Diffusion Policy with Flow Matching",
    "course": "CS 8803 Deep Reinforcement Learning",
    "benchmark": "PushT",
    "experiments": experiments,
    "summary": {
        "ddpm_baseline": {
            "score": ddpm_row['best_score'],
            "latency_ms": ddpm_row['latency_p50_ms'],
            "inference_steps": int(ddpm_row['inference_steps'])
        },
        "best_fm": {
            "score": best_fm['best_score'],
            "latency_ms": best_fm['latency_p50_ms'],
            "inference_steps": int(best_fm['inference_steps']),
            "speedup": round(best_fm['speedup'], 1),
            "accuracy_drop_percent": round(best_fm['accuracy_drop_%'], 1)
        },
        "key_findings": [
            "Flow Matching achieves 13.5x speedup with 8 inference steps",
            "Only 10.4% accuracy drop compared to DDPM baseline",
            "8 steps is optimal: 4 steps too few, 16 steps shows diminishing returns",
            "UNet architecture works well; Transformer fails with FM"
        ]
    }
}

# Save to JSON
output_path = RESULTS_DIR / "consolidated_results.json"
with open(output_path, 'w') as f:
    json.dump(consolidated_results, f, indent=2)

print(f"✓ Results saved to: {output_path}")

# Also save as CSV for easy viewing
csv_path = RESULTS_DIR / "experiment_results.csv"
main_results.to_csv(csv_path, index=False)
print(f"✓ CSV saved to: {csv_path}")

---

## Reproducibility Notes

### How to Reproduce These Results

1. **Environment Setup**:
   ```bash
   conda create -n DPFM python=3.9 -y
   conda activate DPFM
   pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
   pip install -r requirements.txt
   ```

2. **Download Data**:
   ```bash
   mkdir -p data/training && cd data/training
   wget https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip
   unzip pusht.zip
   ```

3. **Train Models**:
   ```bash
   # DDPM Baseline
   python dpfm/train.py --config-name=train_ddpm_unet_hybrid_pusht training.seed=42
   
   # Flow Matching (8 steps)
   python dpfm/train.py --config-name=train_fm_unet_hybrid_image_workspace \
       policy.num_inference_steps=8 training.seed=42
   ```

4. **Evaluate**:
   ```bash
   python dpfm/eval.py --checkpoint_path <path_to_ckpt> --n_eval_episodes 50
   ```

### Hardware Used
- GPU: NVIDIA L40S (48GB)
- Training Time: 6-20 hours per experiment
- Cluster: Georgia Tech PACE Phoenix